Collateral - Island token

Debt - Stable token

Leveraged token - volatile token

$
\text{leverage} = \frac{x}{c} + y + 1 \, ,
$

where

$x = \frac{\text{volatile token value}}{\text{nav}}\, ,$

$c$ - calm part of the island token,

$y = \frac{\text{looped island token value}}{\text{nav}}\, .$

Model expects next input:

In [ ]:
np.array([
    leverage_normed,
    cr_normed,
    leverage_vol_normed,
    leverage_mean_normed,
    leverage_gain_mean_normed,
    vol_token_normed,
    vol_token_mean,
    vol_token_gain_mean_normed,
    current_calm_part,
    calm_part_mean,
    calm_part_gain_mean_normed,
    vol_token_vol_normed,
    calm_part_vol_normed,
    leverage_gain_consistency,
    vol_token_gain_consistency,
    calm_part_gain_consistency
    p_return_normed,
    p_return_abs_normed,
    p_return_square_normed,
    p_return_sigma_normed,
    l_return_normed,
    l_return_abs_normed,
    l_return_square_normed,
    l_return_sigma_normed,
    c_return_normed,
    c_return_abs_normed,
    c_return_square_normed,
    c_return_sigma_normed,
    nav_return_normed,
    nav_return_abs_normed,
    nav_return_square_normed,
    nav_return_sigma_normed,
], dtype=np.float32)

The gain consistency of the corresponding series is defined as the fraction of all series movements whose direction coincides with the sign of the slope (over the horizon):

$$
\text{Gain consistancy} = \frac{1}{T}\sum \{1:\Delta s * \text{slope} > 0\}\,.
$$

Here under vol_token or vol_token_normed asuming $x$.

In [ ]:
norm = top_leverage - bottom_leverage

leverage_normed = (leverage - bottom_leverage)/norm
leverage_mean_normed = (leverage_mean - bottom_leverage)/norm

Collaterization ratio

In [ ]:
cr_normed = 1/(cr)

Here the volatility of the corresponding serieses (not their log) is calculated over the horizon

In [ ]:
leverage_vol_normed = math.tanh(5*leverage_vol/norm)
vol_token_vol_normed = math.tanh(10*vol_token_vol)
calm_part_vol_normed = math.tanh(10*calm_part_vol)

Under the slope we mean the slope of the time series calculated with less square method over the horizon

In [ ]:
def _normalize_slope(self, slope, scale = 2000): # scale = 4000 for vol_token_gain_mean
        alpha = math.atan(scale/self.horizon*min(self.step_count + 1, self.horizon)*slope)
        return (2/math.pi*alpha + 1)/2

Here

$$
p\_ return = \log{\left(\frac{\text{vol token price}_{s-1}}{\text{vol token price}_{s}}\right)}\,
$$
$$
l\_ return = \log{\left(\frac{\text{leverage}_{s-1}}{\text{leverage}_{s}}\right)}\,
$$
$$
nav\_ return = \log{\left(\frac{\text{nav}_{s-1}}{\text{nav}_{s}}\right)}\,
$$
$$
c\_ return = \log{\left(\frac{\text{calm part}_{s-1}}{\text{calm part}_{s}}\right)}\,
$$

returns normilized as follows:

In [ ]:
def _normilize_returns(self):
    self.c_return_normed = (math.tanh(self.c_return/0.03) + 1)*0.5
    self.p_return_normed = (math.tanh(self.p_return/0.009) + 1)*0.5
    self.nav_return_normed = (math.tanh(self.nav_return/0.008) + 1)*0.5
    self.l_return_normed = (math.tanh(self.l_return/0.02) + 1)*0.5

    self.c_return_abs_normed = math.tanh(abs(self.c_return)/0.03)
    self.p_return_abs_normed = math.tanh(abs(self.p_return/0.009))
    self.nav_return_abs_normed = math.tanh(abs(self.nav_return/0.008))
    self.l_return_abs_normed = math.tanh(abs(self.l_return/0.02))

    self.c_return_square_normed = math.tanh((self.c_return/0.03)**2)
    self.p_return_square_normed = math.tanh((self.p_return/0.009)**2)
    self.nav_return_square_normed = math.tanh((self.nav_return/0.008)**2)
    self.l_return_square_normed = math.tanh((self.l_return/0.02)**2)

    self.c_return_sigma_normed = math.tanh(self.c_return_vol/0.015)
    self.p_return_sigma_normed = math.tanh(self.p_return_vol/0.004)
    self.nav_return_sigma_normed = math.tanh(self.nav_return_vol/0.004)
    self.l_return_sigma_normed = math.tanh(self.l_return_vol/0.01)

Model returns:

0 - do nothing

1 - set leverage_normed to 0.5

2 - set leverage_normed to 0.75

3 - set leverage_normed to 0.25

The model assumes that the environment maintains the boundary leverage — that is, the maximum leverage affordable according to the collateralization ratio — above top_leverage.

The model is designed for prices with  return volatility (standard deviation of the $\log(p_t)$ series) of about 0.05–0.08.

WARNING: leverage(and x) slopes, volatilities and $l\_ returns$ are calculeting with ignoring rebalancing shift.